# Comparateur de courses - Carrefour & Auchan

## Objectif du notebook

Créer un système capable de :

- rechercher des produits alimentaires à partir d'ingrédients saisis par l'utilisateur ;
- choisir **Paris** comme ville de référence sur les sites ;
- récupérer uniquement les produits disponibles ;
- récupérer les informations utiles sur Carrefour et Auchan ;
- normaliser les résultats dans un format commun ;
- comparer les prix quand ils sont disponibles ;
- proposer une liste de courses simple selon un budget.

> Le scraping se fait à la demande, à partir des ingrédients saisis. Carrefour peut ouvrir Chrome en visible car le site peut afficher une vérification humaine.

# Étape 1 - Installer les bibliothèques

Décommenter la ligne si nécessaire.

In [22]:
# %pip install selenium beautifulsoup4 pandas

# Étape 2 - Imports et paramètres

## Paramètres utilisateur

Modifier `INGREDIENTS`, `BUDGET_TOTAL` et `MAX_RESULTATS_PAR_SITE` selon le test souhaité.

In [23]:
import json
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import quote_plus, urljoin

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

INGREDIENTS = ["riz", "poulet", "tomates"]
BUDGET_TOTAL = 15.00
MAX_RESULTATS_PAR_SITE = 5
TIMEOUT = 20

VILLE_REFERENCE = "Paris"
CODE_POSTAL_REFERENCE = "75000"

DOSSIER_SORTIE = Path("exports")
DOSSIER_SORTIE.mkdir(exist_ok=True)

# Étape 3 - Fonctions générales

## Nettoyage et conversion des prix

Ces fonctions servent aux deux sites.

In [24]:
def maintenant_iso():
    return datetime.now().isoformat(timespec="seconds")


def nettoyer_texte(texte):
    return re.sub(r"\s+", " ", texte or "").strip()


def prix_vers_float(valeur):
    if valeur is None:
        return None
    texte = str(valeur).replace("€", "").replace(" ", "").replace(" ", "")
    texte = texte.replace(",", ".")
    match = re.search(r"\d+(?:\.\d+)?", texte)
    return float(match.group(0)) if match else None


def extraire_prix_depuis_texte(texte):
    texte = nettoyer_texte(texte)

    # Formats fréquents : "2,55 €", "2.55 €", "2 € 55", "2€55".
    patterns = [
        r"(\d+[,.]\d{1,2})\s*€",
        r"(\d+)\s*€\s*(\d{1,2})",
        r"(\d+)\s*,\s*(\d{1,2})\s*€",
    ]

    for pattern in patterns:
        match = re.search(pattern, texte)
        if not match:
            continue
        if len(match.groups()) == 1:
            return prix_vers_float(match.group(1))
        return prix_vers_float(match.group(1) + "," + match.group(2))

    return None


def extraire_prix_unitaire_depuis_texte(texte):
    texte = nettoyer_texte(texte)
    match = re.search(r"(\d+[,.]\d{1,2})\s*€\s*/\s*(KG|kg|L|l|unité|pièce)", texte)
    if not match:
        match = re.search(r"(\d+)\s*€\s*(\d{1,2})\s*/\s*(KG|kg|L|l|unité|pièce)", texte)
        if not match:
            return None, None
        return prix_vers_float(match.group(1) + "," + match.group(2)), match.group(3).lower()
    return prix_vers_float(match.group(1)), match.group(2).lower()


def creer_driver(headless=False, profil=None):
    options = webdriver.ChromeOptions()
    options.binary_location = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
    options.add_argument("--window-size=1440,1200")
    options.add_argument("--disable-extensions")
    options.add_argument("--no-first-run")
    options.add_argument("--no-default-browser-check")
    if not headless:
        options.add_argument("--start-maximized")
    else:
        options.add_argument("--headless=new")
    if profil:
        options.add_argument("--user-data-dir=" + str(profil))
    return webdriver.Chrome(options=options)


def premier(element, css):
    elements = element.find_elements(By.CSS_SELECTOR, css)
    return elements[0] if elements else None


def texte_css(element, css, defaut=""):
    trouve = premier(element, css)
    return nettoyer_texte(trouve.text) if trouve else defaut


def attr_css(element, css, attribut, defaut=None):
    trouve = premier(element, css)
    return trouve.get_attribute(attribut) if trouve else defaut


def cliquer_si_present(driver, selecteurs, timeout=3):
    for by, valeur in selecteurs:
        try:
            element = WebDriverWait(driver, timeout).until(EC.element_to_be_clickable((by, valeur)))
            driver.execute_script("arguments[0].click();", element)
            time.sleep(1)
            return True
        except Exception:
            continue
    return False


def est_visible_et_cliquable(element):
    try:
        return element.is_displayed() and element.is_enabled()
    except Exception:
        return False

# Étape 4 - Format commun des données

Chaque produit, peu importe le site, sera transformé dans la même structure.

In [25]:
def resultat_normalise(
    retailer,
    query,
    rank,
    product_name=None,
    brand=None,
    category=None,
    description=None,
    quantity=None,
    price=None,
    unit_price=None,
    unit=None,
    promotion=None,
    old_price=None,
    available=None,
    image_url=None,
    product_url=None,
    product_id=None,
    store_name=None,
    store_address=None,
    city=None,
    postal_code=None,
    pickup_mode=None,
    search_url=None,
):
    return {
        "retailer": retailer,
        "store_name": store_name,
        "store_address": store_address,
        "city": city,
        "postal_code": postal_code,
        "pickup_mode": pickup_mode,
        "query": query,
        "rank": rank,
        "product_id": product_id,
        "product_name": product_name,
        "brand": brand,
        "category": category,
        "description": description,
        "quantity": quantity,
        "price": price,
        "unit_price": unit_price,
        "unit": unit,
        "promotion": promotion,
        "old_price": old_price,
        "available": available,
        "image_url": image_url,
        "product_url": product_url,
        "source": retailer,
        "search_url": search_url,
        "collected_at": maintenant_iso(),
        "currency": "EUR",
    }

# Étape 5 - Scraper Carrefour

## Structure Carrefour utilisée

- Recherche : `https://www.carrefour.fr/s?q=...`
- Carte produit : `article[data-testid]`
- Lien fiche : `a.product-card-click-wrapper[href*='/p/']`
- Fiche produit : JSON-LD `script[type='application/ld+json']` contenant `@type = Product`.

## Ville de référence

Le notebook tente de sélectionner **Paris** avant la recherche. Si Carrefour n'affiche pas le panneau de choix de magasin, le scraping continue quand même, mais les produits non disponibles sont filtrés grâce au JSON-LD de la fiche produit.

In [36]:
def extraire_jsonld_product(page_source):
    soup = BeautifulSoup(page_source, "html.parser")
    for script in soup.select("script[type='application/ld+json']"):
        contenu = script.get_text(strip=True)
        if not contenu:
            continue
        try:
            data = json.loads(contenu)
        except json.JSONDecodeError:
            continue

        items = data if isinstance(data, list) else [data]
        for item in items:
            if isinstance(item, dict) and item.get("@type") == "Product":
                return item
    return None


def selectionner_paris_carrefour(driver):
    # Tente de choisir Paris comme zone de drive/livraison sur Carrefour.
    cliquer_si_present(driver, [
        (By.XPATH, "//button[contains(., 'Drive') or contains(@aria-label, 'Drive')]"),
        (By.XPATH, "//button[contains(., 'Livraison') or contains(@aria-label, 'Livraison')]"),
    ], timeout=4)

    champs = driver.find_elements(By.XPATH, "//input[contains(@placeholder, 'ville') or contains(@placeholder, 'postal') or contains(@placeholder, 'adresse') or contains(@aria-label, 'ville') or contains(@aria-label, 'postal')]")
    if not champs:
        return False

    champ = champs[0]
    champ.clear()
    champ.send_keys(VILLE_REFERENCE)
    time.sleep(2)

    cliquer_si_present(driver, [
        (By.XPATH, "//*[self::button or self::li or self::div][contains(normalize-space(.), 'Paris')]"),
        (By.XPATH, "//*[self::button or self::li or self::div][contains(normalize-space(.), '75000')]"),
    ], timeout=4)
    cliquer_si_present(driver, [
        (By.XPATH, "//button[contains(., 'Choisir') or contains(., 'Valider') or contains(., 'Confirmer')]"),
    ], timeout=4)
    return True


def scraper_carrefour(ingredients, max_resultats=5):
    driver = creer_driver(headless=False, profil=None)
    wait = WebDriverWait(driver, TIMEOUT)
    resultats = []

    try:
        driver.get("https://www.carrefour.fr/")
        time.sleep(5)
        selectionner_paris_carrefour(driver)

        for ingredient in ingredients:
            search_url = f"https://www.carrefour.fr/s?q={quote_plus(ingredient)}"
            driver.get(search_url)
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article[data-testid]")))
            cartes = driver.find_elements(By.CSS_SELECTOR, "article[data-testid]")[:max_resultats]

            liens = []
            for rank, carte in enumerate(cartes, start=1):
                lien = attr_css(carte, "a.product-card-click-wrapper[href*='/p/']", "href")
                if lien and lien not in [item[1] for item in liens]:
                    liens.append((rank, lien, carte.get_attribute("data-testid")))

            for rank, product_url, product_id in liens:
                driver.get(product_url)
                time.sleep(2)
                product = extraire_jsonld_product(driver.page_source)
                if not product:
                    continue

                brand = product.get("brand", {})
                if isinstance(brand, dict):
                    brand = brand.get("name")
                image = product.get("image")
                if isinstance(image, dict):
                    image = image.get("url")
                offers = product.get("offers", {}) or {}
                price = offers.get("lowPrice") or offers.get("price")
                availability = str(offers.get("availability", "")).lower()
                available = "instock" in availability if availability else False

                if not available:
                    continue

                resultats.append(resultat_normalise(
                    retailer="Carrefour",
                    query=ingredient,
                    rank=rank,
                    product_id=product.get("gtin13") or product_id,
                    product_name=product.get("name"),
                    brand=brand,
                    description=product.get("description"),
                    quantity=product.get("description"),
                    price=prix_vers_float(price),
                    available=True,
                    image_url=image,
                    product_url=product.get("url") or product_url,
                    city=VILLE_REFERENCE,
                    postal_code=CODE_POSTAL_REFERENCE,
                    search_url=search_url,
                ))
    finally:
        driver.quit()

    return resultats

# Étape 6 - Scraper Auchan

## Structure Auchan utilisée

- Recherche : champ `input.header-search__input[name='text']`.
- Carte produit : `article.product-thumbnail`.
- Lien fiche : `a.product-thumbnail__details-wrapper[href*='/pr-']`.
- Image : `meta[itemprop='image']`.
- Ville/magasin : panneau `Choisir vos courses`, champ `input#journey-search`.

## Ville de référence

Le notebook sélectionne **Paris** avant de rechercher les produits. Une fois le magasin ou service choisi, Auchan affiche les prix dans les cartes produit. Les cartes qui restent sans prix ou avec `Afficher le prix` sont ignorées.

In [27]:
def accepter_ou_refuser_cookies_auchan(driver):
    for css in ["#onetrust-reject-all-handler", "#onetrust-accept-btn-handler", ".onetrust-close-btn-handler"]:
        try:
            bouton = WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.CSS_SELECTOR, css)))
            bouton.click()
            time.sleep(1)
            print(f"Cookies Auchan traités avec : {css}")
            return True
        except TimeoutException:
            continue
    print("Aucun bandeau cookies Auchan détecté.")
    return False


def selectionner_paris_auchan(driver):
    """Sélectionne Paris puis clique sur le premier bouton Choisir de la liste magasin.

    Important : Auchan ne considère le magasin comme choisi qu'après ce clic sur Choisir.
    """
    wait = WebDriverWait(driver, TIMEOUT)
    print("Ouverture du panneau magasin/livraison Auchan...")
    ouvert = cliquer_si_present(driver, [
        (By.CSS_SELECTOR, ".layerTriggerJourneyReminder"),
        (By.CSS_SELECTOR, "button[aria-label='Choisir mon mode de livraison']"),
        (By.XPATH, "//button[contains(., 'Choisir vos courses')]"),
    ], timeout=8)

    if not ouvert:
        print("Panneau magasin non ouvert : aucun bouton de choix trouvé.")
        return False

    champ = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input#journey-search")))
    champ.clear()
    champ.send_keys(VILLE_REFERENCE)
    time.sleep(3)

    suggestions = [e for e in driver.find_elements(By.CSS_SELECTOR, "#search_suggests li, .journey__search-suggests li") if nettoyer_texte(e.text)]
    print(f"Suggestions Paris trouvées : {len(suggestions)}")
    for suggestion in suggestions[:5]:
        print(" -", nettoyer_texte(suggestion.text))

    if not suggestions:
        print("Aucune suggestion Paris trouvée. Le magasin n'est pas sélectionné.")
        return False

    driver.execute_script("arguments[0].click();", suggestions[0])
    time.sleep(6)

    # Le clic obligatoire : bouton "Choisir" du premier magasin/service.
    # Exemple ciblé :
    # <button class="btn btn--small btnJourneySubmit" aria-label="Choisir le mode d'achat : Retrait à Auchan Drive Supermarché Buttes Chaumont - Paris">Choisir</button>
    boutons_choisir = []

    selecteurs_choisir = [
        "button.btnJourneySubmit[aria-label*='Choisir le mode']",
        "button.btnJourneySubmit[aria-label*='Choisir']",
        "button.btnJourneySubmit",
        "button[aria-label*='Choisir le mode']",
        "button[aria-label*='Auchan Drive'][aria-label*='Paris']",
        "//button[contains(@class, 'btnJourneySubmit') and contains(normalize-space(.), 'Choisir')]",
        "//button[contains(@aria-label, 'Choisir le mode') and contains(@aria-label, 'Paris')]",
        "//button[contains(normalize-space(.), 'Choisir') or contains(@aria-label, 'Choisir')]",
    ]

    for selecteur in selecteurs_choisir:
        if selecteur.startswith("//"):
            elements = driver.find_elements(By.XPATH, selecteur)
        else:
            elements = driver.find_elements(By.CSS_SELECTOR, selecteur)
        for bouton in elements:
            if est_visible_et_cliquable(bouton) and bouton not in boutons_choisir:
                boutons_choisir.append(bouton)

    print(f"Boutons 'Choisir' trouvés après Paris : {len(boutons_choisir)}")
    for bouton in boutons_choisir[:5]:
        print(" -", nettoyer_texte(bouton.text), "|", bouton.get_attribute("class"), "|", bouton.get_attribute("aria-label"))

    if boutons_choisir:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", boutons_choisir[0])
        time.sleep(1)
        driver.execute_script("arguments[0].click();", boutons_choisir[0])
        time.sleep(8)
        print("Premier magasin/service Paris choisi avec le bouton Choisir.")
        return True

    print("Aucun bouton Choisir exploitable trouvé après Paris.")
    return False


def analyser_texte_produit_auchan(texte, query, rank, search_url, product_url=None, product_id=None, image_url=None):
    texte = nettoyer_texte(texte)
    lignes = [ligne.strip() for ligne in texte.splitlines() if ligne.strip()]
    lignes_utiles = [ligne for ligne in lignes if not re.fullmatch(r"\(\d+\)", ligne)]

    nom_complet = lignes_utiles[0] if lignes_utiles else "Nom non trouvé"
    quantity = lignes_utiles[1] if len(lignes_utiles) > 1 else None
    morceaux = nom_complet.split(" ", 1)
    brand = morceaux[0] if morceaux else None
    product_name = morceaux[1] if len(morceaux) > 1 else nom_complet

    price = extraire_prix_depuis_texte(texte)
    unit_price, unit = extraire_prix_unitaire_depuis_texte(texte)

    return resultat_normalise(
        retailer="Auchan",
        query=query,
        rank=rank,
        product_id=product_id,
        product_name=product_name,
        brand=brand,
        quantity=quantity,
        price=price,
        unit_price=unit_price,
        unit=unit,
        available=price is not None,
        image_url=image_url,
        product_url=product_url,
        city=VILLE_REFERENCE,
        postal_code=CODE_POSTAL_REFERENCE,
        search_url=search_url,
    )


def extraire_infos_fiche_auchan(driver, product_url, query, rank, search_url, product_id=None, image_url=None, debug=False):
    """Ouvre la fiche produit Auchan, clique Afficher le prix si présent, puis récupère infos/prix."""
    driver.get(product_url)
    time.sleep(5)

    boutons_prix = [b for b in driver.find_elements(By.XPATH, "//button[contains(normalize-space(.), 'Afficher le prix')]") if est_visible_et_cliquable(b)]
    if boutons_prix:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", boutons_prix[0])
        time.sleep(1)
        driver.execute_script("arguments[0].click();", boutons_prix[0])
        time.sleep(4)

    h1 = texte_css(driver, "h1", "Nom non trouvé")
    body_text = nettoyer_texte(driver.find_element(By.TAG_NAME, "body").text)
    price = extraire_prix_depuis_texte(body_text)
    unit_price, unit = extraire_prix_unitaire_depuis_texte(body_text)

    marque = None
    quantity = None
    lignes = [ligne.strip() for ligne in driver.find_element(By.TAG_NAME, "body").text.splitlines() if ligne.strip()]
    if h1 in lignes:
        idx = lignes.index(h1)
        if idx > 0:
            marque = lignes[idx - 1]
        if idx + 1 < len(lignes):
            quantity = lignes[idx + 1]

    if not image_url:
        image_url = attr_css(driver, "meta[itemprop='image']", "content") or attr_css(driver, "img", "src")

    if debug:
        print(f"Fiche {rank} | prix={price} | nom={h1} | url={product_url}")

    if price is None:
        return None

    return resultat_normalise(
        retailer="Auchan",
        query=query,
        rank=rank,
        product_id=product_id,
        product_name=h1,
        brand=marque,
        quantity=quantity,
        price=price,
        unit_price=unit_price,
        unit=unit,
        available=True,
        image_url=image_url,
        product_url=product_url,
        city=VILLE_REFERENCE,
        postal_code=CODE_POSTAL_REFERENCE,
        search_url=search_url,
    )


def parser_carte_auchan(carte, query, rank, search_url, driver=None, debug=False):
    lien = attr_css(carte, "a.product-thumbnail__details-wrapper[href*='/pr-']", "href")
    product_url = urljoin("https://www.auchan.fr/", lien) if lien else None
    texte = texte_css(carte, "a.product-thumbnail__details-wrapper[href*='/pr-']")
    texte_carte = nettoyer_texte(carte.text)
    product_id = carte.get_attribute("data-id")
    image_url = attr_css(carte, "meta[itemprop='image']", "content") or attr_css(carte, "img", "src")

    price = extraire_prix_depuis_texte(texte_carte)
    demande_prix = "afficher le prix" in texte_carte.lower()
    indisponible = "indisponible" in texte_carte.lower()

    if debug:
        print(f"Carte {rank} | prix={price} | afficher_prix={demande_prix} | indisponible={indisponible} | texte={texte_carte[:140]}")

    if indisponible:
        return None

    # Si la carte n'a pas de prix ou demande Afficher le prix, on ouvre la fiche.
    if (price is None or demande_prix) and driver and product_url:
        return extraire_infos_fiche_auchan(
            driver=driver,
            product_url=product_url,
            query=query,
            rank=rank,
            search_url=search_url,
            product_id=product_id,
            image_url=image_url,
            debug=debug,
        )

    if price is None:
        return None

    produit = analyser_texte_produit_auchan(
        texte=texte,
        query=query,
        rank=rank,
        search_url=search_url,
        product_url=product_url,
        product_id=product_id,
        image_url=image_url,
    )
    produit["price"] = price
    produit["available"] = True
    return produit


def scraper_auchan(ingredients, max_resultats=5, debug=True):
    driver = creer_driver(headless=False, profil=None)
    wait = WebDriverWait(driver, TIMEOUT)
    resultats = []

    try:
        driver.get("https://www.auchan.fr/")
        accepter_ou_refuser_cookies_auchan(driver)
        paris_ok = selectionner_paris_auchan(driver)
        print("Sélection Paris Auchan :", paris_ok)

        for ingredient in ingredients:
            print()
            print(f"Recherche Auchan : {ingredient}")
            champ = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input.header-search__input[name='text']")))
            champ.clear()
            champ.send_keys(ingredient)
            champ.send_keys(Keys.ENTER)
            time.sleep(6)
            search_url = driver.current_url
            print("URL résultats :", search_url)

            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.product-thumbnail")))
            cartes = driver.find_elements(By.CSS_SELECTOR, "article.product-thumbnail")
            print(f"Cartes trouvées : {len(cartes)}")

            # On capture les URLs avant d'ouvrir les fiches, car l'ouverture change la page courante.
            infos_cartes = []
            for rank, carte in enumerate(cartes, start=1):
                infos_cartes.append({
                    "rank": rank,
                    "html": carte.get_attribute("outerHTML"),
                    "text": carte.text,
                })

            rank_resultats = 0
            for info in infos_cartes:
                # Recrée un mini parseur BeautifulSoup pour conserver les infos de carte après navigation.
                soup = BeautifulSoup(info["html"], "html.parser")
                lien_tag = soup.select_one("a.product-thumbnail__details-wrapper[href*='/pr-']")
                product_url = urljoin("https://www.auchan.fr/", lien_tag.get("href")) if lien_tag else None
                image_tag = soup.select_one("meta[itemprop='image']")
                image_url = image_tag.get("content") if image_tag else None
                product_id = soup.select_one("article.product-thumbnail")
                product_id = product_id.get("data-id") if product_id else None
                texte_carte = nettoyer_texte(info["text"])
                price = extraire_prix_depuis_texte(texte_carte)
                demande_prix = "afficher le prix" in texte_carte.lower()
                indisponible = "indisponible" in texte_carte.lower()

                if debug and info["rank"] <= 10:
                    print(f"Carte {info['rank']} | prix={price} | afficher_prix={demande_prix} | indisponible={indisponible} | url={product_url}")

                if indisponible or not product_url:
                    continue

                if price is None or demande_prix:
                    produit = extraire_infos_fiche_auchan(
                        driver=driver,
                        product_url=product_url,
                        query=ingredient,
                        rank=info["rank"],
                        search_url=search_url,
                        product_id=product_id,
                        image_url=image_url,
                        debug=debug and info["rank"] <= 10,
                    )
                    driver.get(search_url)
                    time.sleep(2)
                else:
                    produit = analyser_texte_produit_auchan(
                        texte=info["text"],
                        query=ingredient,
                        rank=info["rank"],
                        search_url=search_url,
                        product_url=product_url,
                        product_id=product_id,
                        image_url=image_url,
                    )

                if produit and produit.get("price") is not None:
                    produit["available"] = True
                    resultats.append(produit)
                    rank_resultats += 1
                if rank_resultats >= max_resultats:
                    break

            print(f"Produits disponibles avec prix gardés pour {ingredient} : {rank_resultats}")
    finally:
        driver.quit()

    return resultats

# Étape 7 - Lancer le scraping

## Conseil d'exécution

Lancer d'abord Auchan seul, puis Carrefour. Carrefour ouvre Chrome en visible et peut demander une vérification humaine au premier lancement.

In [28]:
resultats_auchan = scraper_auchan(INGREDIENTS, MAX_RESULTATS_PAR_SITE)
df_auchan = pd.DataFrame(resultats_auchan)
df_auchan.head(10)

Cookies Auchan traités avec : #onetrust-reject-all-handler
Ouverture du panneau magasin/livraison Auchan...
Suggestions Paris trouvées : 26
 - Paris 75000
 - Paris 75001
 - Paris 75002
 - Paris 75003
 - Paris 75004
Boutons 'Choisir' trouvés après Paris : 20
 - Choisir | btn btn--small btnJourneySubmit | Choisir le mode d'achat : Retrait à Auchan Drive Supermarché Buttes Chaumont - Paris
 - Choisir | btn btn--small btnJourneySubmit | Choisir le mode d'achat : Retrait à Auchan Drive Hypermarché KREMLIN BICETRE
 - Choisir | btn btn--small btnJourneySubmit | Choisir le mode d'achat : Retrait à Auchan Click&Collect Supermarché Tolbiac-Paris
 - Choisir | btn btn--small btnJourneySubmit | Choisir le mode d'achat : Retrait à Auchan Click&Collect Supermarché PARIS ALESIA
 - Choisir | btn btn--small btnJourneySubmit | Choisir le mode d'achat : Retrait à Auchan Supermarché Click&Collect Reuilly-Paris
Premier magasin/service Paris choisi avec le bouton Choisir.
Sélection Paris Auchan : True

Reche

,retailer,store_name,store_address,city,postal_code,pickup_mode,query,rank,product_id,product_name,...,unit,promotion,old_price,available,image_url,product_url,source,search_url,collected_at,currency
0,Auchan,None,None,Paris,75000,None,riz,1,5326353a-e633-474d-80b9-fb1625a4da1c,"AUCHAN Riz basmati prêt en 11 min 1kg 3,07€ / ...",...,kg,None,None,True,https://cdn.auchan.fr/media/A02199207060000620...,https://www.auchan.fr/auchan-riz-basmati-pret-...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
1,Auchan,None,None,Paris,75000,None,riz,2,e1445254-4cce-4a39-9f95-548f29e6d005,"AUCHAN Riz basmati prêt en 11 min 500g 3,18€ /...",...,kg,None,None,True,https://cdn.auchan.fr/media/A02199704100000667...,https://www.auchan.fr/auchan-riz-basmati-pret-...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
2,Auchan,None,None,Paris,75000,None,riz,3,a0141661-c944-44f4-ad9b-7c289acb2e5e,"TAUREAU AILE Riz basmati du Penjab 1kg 4,74€ /...",...,kg,None,None,True,https://cdn.auchan.fr/media/S01000000041CGXPRI...,https://www.auchan.fr/taureau-aile-riz-basmati...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
3,Auchan,None,None,Paris,75000,None,riz,4,72cee35b-7cb4-4e30-b58a-da7f905b59d0,clients adorent* (26) AUCHAN BIO Riz basmati p...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02200608300004300...,https://www.auchan.fr/auchan-bio-riz-basmati-p...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
4,Auchan,None,None,Paris,75000,None,riz,5,8d1b1a59-665f-458a-b156-74b2896ece6d,clients adorent* (11) AUCHAN Riz basmati sache...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02200205310007565...,https://www.auchan.fr/auchan-riz-basmati-sache...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
5,Auchan,None,None,Paris,75000,None,poulet,1,e034c694-8122-45b6-8fdb-b60d7379653f,choc (20) DUC Filets de poulet blanc 720g 4 pi...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02201902190001106...,https://www.auchan.fr/duc-filets-de-poulet-bla...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR
6,Auchan,None,None,Paris,75000,None,poulet,2,8ac26918-927e-45e0-827d-12fc9a3a0352,choc (66) ROTISSERIE Poulet rôti fermier cuit ...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02200311270000409...,https://www.auchan.fr/rotisserie-poulet-roti-f...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR
7,Auchan,None,None,Paris,75000,None,poulet,3,40ade95b-9772-494a-afee-8a9f35419817,choc (10) LYRE CULTIVONS LE BON Poulet jaune f...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02199303040000372...,https://www.auchan.fr/lyre-cultivons-le-bon-po...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR
8,Auchan,None,None,Paris,75000,None,poulet,4,7163aaf5-036c-4785-9f52-522986ecb6f6,(7) FLEURY MICHON Blanc de poulet rôti à la br...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02201309100006567...,https://www.auchan.fr/fleury-michon-blanc-de-p...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR
9,Auchan,None,None,Paris,75000,None,poulet,5,d00b094e-57c3-45f7-82ec-9dbb49d72a4c,choc (5) LYRE CULTIVONS LE BON Poulet fermier ...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02198904030000367...,https://www.auchan.fr/lyre-cultivons-le-bon-po...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR


In [38]:
resultats_carrefour = scraper_carrefour(INGREDIENTS, MAX_RESULTATS_PAR_SITE)
df_carrefour = pd.DataFrame(resultats_carrefour)
df_carrefour.head(10)

""


# Étape 8 - Fusionner et normaliser les résultats

Tous les produits sont regroupés dans un seul tableau commun.

In [30]:
SCHEMA_PRODUITS = [
    "retailer", "store_name", "store_address", "city", "postal_code", "pickup_mode",
    "query", "rank", "product_id", "product_name", "brand", "category", "description",
    "quantity", "price", "unit_price", "unit", "promotion", "old_price", "available",
    "image_url", "product_url", "source", "search_url", "collected_at", "currency"
]

frames = []
for frame_name in ["df_carrefour", "df_auchan"]:
    if frame_name in globals():
        frame = globals()[frame_name]
        if isinstance(frame, pd.DataFrame):
            frames.append(frame)

if frames:
    df_produits = pd.concat(frames, ignore_index=True)
else:
    df_produits = pd.DataFrame()

# Garantit que les colonnes existent même si aucun produit n'a été récupéré.
for colonne in SCHEMA_PRODUITS:
    if colonne not in df_produits.columns:
        df_produits[colonne] = pd.NA

df_produits = df_produits[SCHEMA_PRODUITS]

if df_produits.empty:
    print("Aucun produit récupéré pour le moment. Vérifiez les cellules de scraping Auchan/Carrefour avant de continuer.")

df_produits

,retailer,store_name,store_address,city,postal_code,pickup_mode,query,rank,product_id,product_name,...,unit,promotion,old_price,available,image_url,product_url,source,search_url,collected_at,currency
0,Auchan,None,None,Paris,75000,None,riz,1,5326353a-e633-474d-80b9-fb1625a4da1c,"AUCHAN Riz basmati prêt en 11 min 1kg 3,07€ / ...",...,kg,None,None,True,https://cdn.auchan.fr/media/A02199207060000620...,https://www.auchan.fr/auchan-riz-basmati-pret-...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
1,Auchan,None,None,Paris,75000,None,riz,2,e1445254-4cce-4a39-9f95-548f29e6d005,"AUCHAN Riz basmati prêt en 11 min 500g 3,18€ /...",...,kg,None,None,True,https://cdn.auchan.fr/media/A02199704100000667...,https://www.auchan.fr/auchan-riz-basmati-pret-...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
2,Auchan,None,None,Paris,75000,None,riz,3,a0141661-c944-44f4-ad9b-7c289acb2e5e,"TAUREAU AILE Riz basmati du Penjab 1kg 4,74€ /...",...,kg,None,None,True,https://cdn.auchan.fr/media/S01000000041CGXPRI...,https://www.auchan.fr/taureau-aile-riz-basmati...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
3,Auchan,None,None,Paris,75000,None,riz,4,72cee35b-7cb4-4e30-b58a-da7f905b59d0,clients adorent* (26) AUCHAN BIO Riz basmati p...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02200608300004300...,https://www.auchan.fr/auchan-bio-riz-basmati-p...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
4,Auchan,None,None,Paris,75000,None,riz,5,8d1b1a59-665f-458a-b156-74b2896ece6d,clients adorent* (11) AUCHAN Riz basmati sache...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02200205310007565...,https://www.auchan.fr/auchan-riz-basmati-sache...,Auchan,https://www.auchan.fr/epicerie-salee/pates-riz...,2026-06-04T16:02:05,EUR
5,Auchan,None,None,Paris,75000,None,poulet,1,e034c694-8122-45b6-8fdb-b60d7379653f,choc (20) DUC Filets de poulet blanc 720g 4 pi...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02201902190001106...,https://www.auchan.fr/duc-filets-de-poulet-bla...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR
6,Auchan,None,None,Paris,75000,None,poulet,2,8ac26918-927e-45e0-827d-12fc9a3a0352,choc (66) ROTISSERIE Poulet rôti fermier cuit ...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02200311270000409...,https://www.auchan.fr/rotisserie-poulet-roti-f...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR
7,Auchan,None,None,Paris,75000,None,poulet,3,40ade95b-9772-494a-afee-8a9f35419817,choc (10) LYRE CULTIVONS LE BON Poulet jaune f...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02199303040000372...,https://www.auchan.fr/lyre-cultivons-le-bon-po...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR
8,Auchan,None,None,Paris,75000,None,poulet,4,7163aaf5-036c-4785-9f52-522986ecb6f6,(7) FLEURY MICHON Blanc de poulet rôti à la br...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02201309100006567...,https://www.auchan.fr/fleury-michon-blanc-de-p...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR
9,Auchan,None,None,Paris,75000,None,poulet,5,d00b094e-57c3-45f7-82ec-9dbb49d72a4c,choc (5) LYRE CULTIVONS LE BON Poulet fermier ...,...,kg,None,None,True,https://cdn.auchan.fr/media/A02198904030000367...,https://www.auchan.fr/lyre-cultivons-le-bon-po...,Auchan,https://www.auchan.fr/recherche?text=poulet,2026-06-04T16:02:14,EUR


# Étape 9 - Calculer un score de pertinence

Le score favorise les produits dont le nom contient l'ingrédient recherché et qui ont un prix disponible.

In [31]:
def calculer_score(row):
    score = 0
    query = str(row.get("query", "")).lower()
    name = str(row.get("product_name", "")).lower()
    if query and query in name:
        score += 70
    if pd.notna(row.get("price")):
        score += 20
    if row.get("available") is True:
        score += 10
    return score


if "relevance_score" not in df_produits.columns:
    df_produits["relevance_score"] = pd.Series(dtype="float")

if df_produits.empty:
    print("Impossible de calculer un score : df_produits est vide.")
else:
    df_produits["relevance_score"] = df_produits.apply(calculer_score, axis=1)

colonnes_tri = ["query", "relevance_score", "price"]
colonnes_tri = [col for col in colonnes_tri if col in df_produits.columns]

if colonnes_tri:
    df_produits.sort_values(colonnes_tri, ascending=[True, False, True][:len(colonnes_tri)])
else:
    df_produits

# Étape 10 - Comparer les prix disponibles

Les lignes sans prix ou indisponibles ont été filtrées pendant le scraping. Cette étape compare donc uniquement les produits disponibles avec prix exploitable.

In [32]:
df_prix = df_produits[(df_produits["available"] == True) & df_produits["price"].notna()].copy()

comparaison_prix = (
    df_prix.sort_values(["query", "price", "relevance_score"], ascending=[True, True, False])
    .groupby("query")
    .head(5)
)

comparaison_prix[["query", "retailer", "product_name", "brand", "quantity", "price", "unit_price", "unit", "product_url"]]

,query,retailer,product_name,brand,quantity,price,unit_price,unit,product_url
7,poulet,Auchan,choc (10) LYRE CULTIVONS LE BON Poulet jaune f...,Prix,None,4.49,4.49,kg,https://www.auchan.fr/lyre-cultivons-le-bon-po...
9,poulet,Auchan,choc (5) LYRE CULTIVONS LE BON Poulet fermier ...,Prix,None,5.85,5.85,kg,https://www.auchan.fr/lyre-cultivons-le-bon-po...
5,poulet,Auchan,choc (20) DUC Filets de poulet blanc 720g 4 pi...,Prix,None,9.71,9.71,kg,https://www.auchan.fr/duc-filets-de-poulet-bla...
6,poulet,Auchan,choc (66) ROTISSERIE Poulet rôti fermier cuit ...,Prix,None,11.89,11.89,kg,https://www.auchan.fr/rotisserie-poulet-roti-f...
8,poulet,Auchan,(7) FLEURY MICHON Blanc de poulet rôti à la br...,Sponsorisé,None,18.75,18.75,kg,https://www.auchan.fr/fleury-michon-blanc-de-p...
0,riz,Auchan,"AUCHAN Riz basmati prêt en 11 min 1kg 3,07€ / ...",(51),None,3.07,3.07,kg,https://www.auchan.fr/auchan-riz-basmati-pret-...
1,riz,Auchan,"AUCHAN Riz basmati prêt en 11 min 500g 3,18€ /...",(51),None,3.18,3.18,kg,https://www.auchan.fr/auchan-riz-basmati-pret-...
3,riz,Auchan,clients adorent* (26) AUCHAN BIO Riz basmati p...,Nos,None,3.56,3.56,kg,https://www.auchan.fr/auchan-bio-riz-basmati-p...
2,riz,Auchan,"TAUREAU AILE Riz basmati du Penjab 1kg 4,74€ /...",(4),None,4.74,4.74,kg,https://www.auchan.fr/taureau-aile-riz-basmati...
4,riz,Auchan,clients adorent* (11) AUCHAN Riz basmati sache...,Nos,None,5.04,5.04,kg,https://www.auchan.fr/auchan-riz-basmati-sache...


# Étape 11 - Proposer une liste de courses selon le budget

Pour chaque ingrédient, on sélectionne le produit disponible le moins cher parmi ceux qui ont un prix.

In [33]:
def generer_liste_courses(df, budget):
    choix = []
    total = 0.0

    for ingredient in INGREDIENTS:
        candidats = df[
            (df["query"] == ingredient)
            & (df["available"] == True)
            & df["price"].notna()
        ].copy()
        candidats = candidats.sort_values(["price", "relevance_score"], ascending=[True, False])
        if candidats.empty:
            continue

        produit = candidats.iloc[0].to_dict()
        if total + float(produit["price"]) <= budget:
            choix.append(produit)
            total += float(produit["price"])

    return pd.DataFrame(choix), round(total, 2)


df_liste_courses, total_courses = generer_liste_courses(df_produits, BUDGET_TOTAL)
print(f"Total estimé : {total_courses:.2f} € / Budget : {BUDGET_TOTAL:.2f} €")
df_liste_courses[["query", "retailer", "product_name", "brand", "quantity", "price", "product_url"]] if not df_liste_courses.empty else df_liste_courses

Total estimé : 9.55 € / Budget : 15.00 €


,query,retailer,product_name,brand,quantity,price,product_url
0,riz,Auchan,"AUCHAN Riz basmati prêt en 11 min 1kg 3,07€ / ...",(51),None,3.07,https://www.auchan.fr/auchan-riz-basmati-pret-...
1,poulet,Auchan,choc (10) LYRE CULTIVONS LE BON Poulet jaune f...,Prix,None,4.49,https://www.auchan.fr/lyre-cultivons-le-bon-po...
2,tomates,Auchan,de saison ! (31) Tomates rondes prix bas Maroc...,C'est,None,1.99,https://www.auchan.fr/tomates-rondes-prix-bas/...


# Étape 12 - Proposer un menu simple

Le menu est généré à partir des ingrédients saisis.

In [34]:
def proposer_menu(ingredients):
    texte_ingredients = ", ".join(ingredients)
    return {
        "nom_menu": f"Plat simple avec {texte_ingredients}",
        "ingredients": ingredients,
        "idee": "Cuire les féculents, ajouter la protéine, puis incorporer les légumes avec un assaisonnement simple.",
    }


menu = proposer_menu(INGREDIENTS)
menu

{'nom_menu': 'Plat simple avec riz, poulet, tomates',
 'ingredients': ['riz', 'poulet', 'tomates'],
 'idee': 'Cuire les féculents, ajouter la protéine, puis incorporer les légumes avec un assaisonnement simple.'}

# Étape 13 - Exporter les résultats

Les exports sont créés en CSV et JSON.

In [35]:
fichier_produits_csv = DOSSIER_SORTIE / "produits_normalises.csv"
fichier_produits_json = DOSSIER_SORTIE / "produits_normalises.json"
fichier_liste_csv = DOSSIER_SORTIE / "liste_courses.csv"
fichier_menu_json = DOSSIER_SORTIE / "menu_simple.json"

df_produits.to_csv(fichier_produits_csv, index=False, encoding="utf-8-sig")
df_produits.to_json(fichier_produits_json, orient="records", force_ascii=False, indent=2)

if not df_liste_courses.empty:
    df_liste_courses.to_csv(fichier_liste_csv, index=False, encoding="utf-8-sig")

fichier_menu_json.write_text(json.dumps(menu, ensure_ascii=False, indent=2), encoding="utf-8")

print("Exports créés :")
print(fichier_produits_csv.resolve())
print(fichier_produits_json.resolve())
print(fichier_liste_csv.resolve())
print(fichier_menu_json.resolve())

Exports créés :
C:\TCHIKSON\Webscrapping\comparateur_courses\exports\produits_normalises.csv
C:\TCHIKSON\Webscrapping\comparateur_courses\exports\produits_normalises.json
C:\TCHIKSON\Webscrapping\comparateur_courses\exports\liste_courses.csv
C:\TCHIKSON\Webscrapping\comparateur_courses\exports\menu_simple.json


# Étape 14 - Points à améliorer

## Pour une version complète

- Choisir explicitement un magasin Paris précis au lieu du premier résultat proposé par Auchan.
- Ajouter plusieurs magasins par ville et comparer les prix par magasin.
- Ajouter d'autres enseignes.
- Améliorer le score de pertinence avec des synonymes et catégories.
- Ajouter Nutri-score, allergènes et valeurs nutritionnelles quand disponibles.
- Nettoyer les anciens profils Chrome si le disque est plein.